In [ ]:
import numpy as np
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from find_ipctk import ipctk
from ipctk import *

In [ ]:
V = np.array([[-1, -0.5], [0, 0], [1, -0.5]])
E = np.array([[0, 1], [1, 2]])
ls = np.linalg.norm(V[E[:, 1]] - V[E[:, 0]], axis=1)
print("Edge lengths:", ls)
ws = np.array([ls[0]/2, ls[0]/2+ls[1]/2, ls[1]/2])

In [ ]:
x, y = np.meshgrid(np.arange(-2, 2, 0.01), np.arange(-2, 2, 0.01))
u, v = np.empty(x.shape), np.empty(x.shape)

dhat = 0.5
dhat2 = dhat**2

for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        grad = np.zeros(6)
        prev_dmin = np.inf
        for e in E:
            d2 = point_edge_distance(
                np.array([x[i, j], y[i, j]]), V[e[0]], V[e[1]])
            if d2 < dhat2:
                s = 2 * (d2/dhat2 - 1) / dhat2 * 1 / (2 * np.sqrt(d2))
                grad += s * point_edge_distance_gradient(
                    np.array([x[i, j], y[i, j]]), V[e[0]], V[e[1]])

        d2 = point_point_distance(np.array([x[i, j], y[i, j]]), V[1])
        if d2 < dhat2:
            s = 2 * (d2/dhat2 - 1) / dhat2 * 1 / (2 * np.sqrt(d2))
            grad[:2] -= s * point_point_distance_gradient(
                np.array([x[i, j], y[i, j]]), V[1])[:2]

        u[i, j] = -grad[0]
        v[i, j] = -grad[1]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("u (gradient x-component)", "v (gradient y-component)")
)

for j in range(2):
    fig.add_trace(
        go.Scatter(x=np.vstack(V[E])[:, 0], y=np.vstack(V[E])[:, 1]),
        row=1, col=j + 1
    )
    fig.add_trace(
        go.Heatmap(z=(u if j == 0 else v).flatten(), x=x.flatten(),
                   y=y.flatten(), colorscale="RdBu"),
        row=1, col=j + 1
    )

# Share a common color scale for both heatmaps
color_min = min(u.min(), v.min())
color_max = max(u.max(), v.max())

fig.update_layout(
    width=1200,
    height=635,
    showlegend=False,
    coloraxis=dict(
        colorscale="RdBu",
        cmin=color_min,
        cmax=color_max,
        colorbar=dict(title="Value")
    )
)
# Update both heatmaps to use the shared coloraxis
fig.data[1].update(coloraxis="coloraxis")
fig.data[3].update(coloraxis="coloraxis")

fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
fig.show()

In [ ]:
# Normalize u and v to [0, 1] for color mapping
u_norm = ((u - u.min()) / (u.max() - u.min()))[::-1, :]
v_norm = ((v - v.min()) / (v.max() - v.min()))[::-1, :]
# u_norm = (abs(u) / np.max(abs(u)))[::-1, :]  # Normalize u to [0, 1]
# v_norm = (abs(v) / np.max(abs(v)))[::-1, :]  # Normalize v to [0, 1]

# Create RGB image: u in red, v in green, blue is zero
rgb_img = np.stack([u_norm, np.zeros_like(u_norm), v_norm], axis=-1)

fig = go.Figure(go.Image(z=(rgb_img * 255).astype(np.uint8)))
fig.update_layout(
    width=1200,
    height=1200,
    title="u (red) and v (green) encoded in RGB",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False)
)
fig.show()